<a href="https://colab.research.google.com/github/mohammedh897/Prediction-of-Product-Sales/blob/main/Project_1_Part_5_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# load data
import pandas as pd
path='/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week05/Data/sales_predictions_2023 (1).csv'
df = pd.read_csv(path)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8523 entries, 0 to 8522
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Identifier            8523 non-null   object 
 1   Item_Weight                7060 non-null   float64
 2   Item_Fat_Content           8523 non-null   object 
 3   Item_Visibility            8523 non-null   float64
 4   Item_Type                  8523 non-null   object 
 5   Item_MRP                   8523 non-null   float64
 6   Outlet_Identifier          8523 non-null   object 
 7   Outlet_Establishment_Year  8523 non-null   int64  
 8   Outlet_Size                6113 non-null   object 
 9   Outlet_Location_Type       8523 non-null   object 
 10  Outlet_Type                8523 non-null   object 
 11  Item_Outlet_Sales          8523 non-null   float64
dtypes: float64(4), int64(1), object(7)
memory usage: 799.2+ KB


,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


### Part 5: Sales Prediction Project - Data Loading and Preprocessing

Reloading the original dataset to ensure no data leakage and starting the cleaning process.

#### Step 1: Handle Duplicates and Inconsistent Categorical Data

### Handle Duplicates

In [13]:
duplicates = df.duplicated().sum()
duplicates

np.int64(0)

### Handle Inconsistent Categorical Data


In [14]:
categorical_cols = df.select_dtypes(include='object').columns
categorical_cols

Index(['Item_Identifier', 'Item_Fat_Content', 'Item_Type', 'Outlet_Identifier',
       'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type'],
      dtype='object')

In [15]:
df['Item_Fat_Content'].value_counts()

,count
Item_Fat_Content,
Low Fat,5089
Regular,2889
LF,316
reg,117
low fat,112


In [17]:
#Fix inconsistent categories in Item_Fat_Content
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'LF': 'low fat',
    'Low Fat': 'low fat',
    'low fat': 'low fat',
    'Reg': 'regular',
    'reg': 'regular',
    'Regular': 'regular'
})


In [18]:
# Check value counts in Item_Fat_Content after cleaning

df['Item_Fat_Content'].value_counts()

,count
Item_Fat_Content,
low fat,5517
regular,3006


#### Step 2: Identify Features (X) and Target (y), Drop 'Item_Identifier'

In [23]:
# Define target variable
y = df['Item_Outlet_Sales']

# Define features, dropping 'Item_Identifier' and the target variable
X = df.drop(columns=['Item_Identifier', 'Item_Outlet_Sales'])


#### Step 3: Perform Train-Test Split

In [24]:
from sklearn.model_selection import train_test_split
# Perform 75/25 train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

Identify each feature as numerical, ordinal, or nominal.

In [25]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6392 entries, 4776 to 7270
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Item_Weight                5285 non-null   float64
 1   Item_Fat_Content           6392 non-null   object 
 2   Item_Visibility            6392 non-null   float64
 3   Item_Type                  6392 non-null   object 
 4   Item_MRP                   6392 non-null   float64
 5   Outlet_Identifier          6392 non-null   object 
 6   Outlet_Establishment_Year  6392 non-null   int64  
 7   Outlet_Size                4580 non-null   object 
 8   Outlet_Location_Type       6392 non-null   object 
 9   Outlet_Type                6392 non-null   object 
dtypes: float64(3), int64(1), object(6)
memory usage: 549.3+ KB


In [26]:
# Checking object columns
X_train.select_dtypes('object')

,Item_Fat_Content,Item_Type,Outlet_Identifier,Outlet_Size,Outlet_Location_Type,Outlet_Type
4776,low fat,Household,OUT018,Medium,Tier 3,Supermarket Type2
7510,regular,Snack Foods,OUT018,Medium,Tier 3,Supermarket Type2
5828,regular,Meat,OUT049,Medium,Tier 1,Supermarket Type1
5327,low fat,Baking Goods,OUT035,Small,Tier 2,Supermarket Type1
4810,low fat,Frozen Foods,OUT045,NaN,Tier 2,Supermarket Type1
...,...,...,...,...,...,...
5734,regular,Fruits and Vegetables,OUT010,NaN,Tier 3,Grocery Store
5191,low fat,Frozen Foods,OUT017,NaN,Tier 2,Supermarket Type1
5390,low fat,Health and Hygiene,OUT045,NaN,Tier 2,Supermarket Type1
860,low fat,Snack Foods,OUT017,NaN,Tier 2,Supermarket Type1


In [27]:
X_train['Item_Fat_Content'].value_counts(dropna=False)

,count
Item_Fat_Content,
low fat,4129
regular,2263


In [28]:
X_train['Item_Type'].value_counts(dropna=False)

,count
Item_Type,
Fruits and Vegetables,948
Snack Foods,906
Household,695
Frozen Foods,632
Dairy,507
Canned,481
Baking Goods,478
Health and Hygiene,390
Soft Drinks,331


In [29]:
X_train['Outlet_Identifier'].value_counts(dropna=False)

,count
Outlet_Identifier,
OUT027,723
OUT035,709
OUT018,704
OUT045,699
OUT017,698
OUT046,695
OUT013,689
OUT049,676
OUT010,415


In [30]:
X_train['Outlet_Size'].value_counts(dropna=False)

,count
Outlet_Size,
Medium,2103
NaN,1812
Small,1788
High,689


In [31]:
X_train['Outlet_Location_Type'].value_counts(dropna=False)

,count
Outlet_Location_Type,
Tier 3,2531
Tier 2,2106
Tier 1,1755


In [32]:
X_train['Outlet_Type'].value_counts(dropna=False)

,count
Outlet_Type,
Supermarket Type1,4166
Grocery Store,799
Supermarket Type3,723
Supermarket Type2,704


## Features:

- Ordinal: Outlet_Size, Outlet_Location_Type
- Categorical: Item_Fat_Content, Item_Type,  Outlet_Identifier, Outlet_Type.
- The remaining features are numeric.
Create 3 pipelines (one for numeric, ordinal, and categorical features).


#### Step 4: Create Preprocessing Object (ColumnTransformer)

Imputation will be part of the preprocessing pipeline and will be applied *after* the train-test split.

In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline

# Identify numerical features
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Define ordinal features and their explicit orderings
ordinal_features = ['Outlet_Size', 'Outlet_Location_Type']
outlet_size_order = ['Small', 'Medium', 'High']
outlet_location_type_order = ['Tier 1', 'Tier 2', 'Tier 3']

# Identify nominal features (remaining object columns after excluding ordinal ones)
all_object_features = X.select_dtypes(include=['object']).columns.tolist()
nominal_features = [col for col in all_object_features if col not in ordinal_features]

# Define preprocessing steps for numerical features
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # Impute missing numerical values with the mean
    ('scaler', StandardScaler()) # Scale numerical features
])

# Define preprocessing steps for ordinal features
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute missing ordinal values with the most frequent
    ('ordinal', OrdinalEncoder(categories=[outlet_size_order, outlet_location_type_order], handle_unknown='use_encoded_value', unknown_value=-1)), # Ordinal encode with specified order
    ('scaler', StandardScaler()) # Scale ordinal features after encoding
])

# Define preprocessing steps for nominal features
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Impute missing nominal values with the most frequent
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # One-hot encode nominal features
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, nominal_features)
    ])
